# viva-Mgen: reproducing the Karr 2012 M. genitalium whole-cell model, one study per figure

_Investigation `mgen` — coder reproduction notebook._

**Question.** Can the core cellular processes of the Karr et al. 2012 Mycoplasma genitalium
whole-cell model be re-expressed as native process-bigraph Processes —
composed through shared cell-variable stores — and reproduce the central
quantitative results of each figure of the paper?

A showcase investigation that reproduces the landmark Karr 2012 whole-cell
model of M. genitalium as native process-bigraph processes, with one study
for each of the paper's seven figures. It demonstrates that the model's
central results — 9 h doubling, protein-dominant composition, decoupled
gene expression, emergent cell-cycle regulation, ATP/GTP energy allocation,
and metabolic gene essentiality — are recoverable from a compact, genuinely
mechanistic reimplementation, without the original's MATLAB code, MySQL
knowledge base, or cluster-scale simulation.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-Mgen/viva-Mgen').is_dir():
    REPO = Path('/home/runner/work/viva-Mgen/viva-Mgen')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_mgen.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Fig 1 — Whole-cell integration: six submodels wired through shared cell variables (`fig1-architecture`)

**Question.** Does viva-Mgen actually reproduce Fig 1's central architectural claim — that
the whole-cell model is a set of independent submodels *integrated* through a
shared set of cell variables — as a real, runnable composite rather than a
drawing? Do all six submodels compose, share stores, and run as one integrated
cell?

**Objective.** Build the integrated composite, derive the process↔cell-variable wiring matrix
directly from the composite document (a genuine Fig 1B-style diagram), count
the submodels and the shared/coupled stores, and run the integrated cell to
confirm it composes and stays viable.

**Hypothesis.** Building the fig1_architecture composite should wire exactly six submodels
(metabolism, mass, transcription, translation, rna_decay, protein_decay) to a
common set of bigraph stores such that several cell variables are touched by
more than one submodel (the coupling that makes it "integrated"), and running
the composite for 20 min should keep the cell viable (growth_fraction ≈ 1) and
growing (mass increasing) — reproducing the integrated-architecture picture of
Fig 1A/1B.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig1-architecture ===
STUDY = 'fig1-architecture'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**wiring-matrix**


In [ ]:
# wiring-matrix
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**cell-dashboard**


In [ ]:
# cell-dashboard
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| architecture-integrates-six-submodels | kind=derived_scalar field=n_processes | op range low 6 high 6 provenance {'kind': 'model', 'note': 'Reduced viva-Mgen cell = six submodels (metabolism, mass, transcription, translation, rna_decay, protein_decay); Fig 1A. Achieved 6.'} |
| submodels-are-coupled | kind=derived_scalar field=n_stores_coupled | op range low 2 high 100 provenance {'kind': 'theory', 'note': 'Integration = communication through shared cell variables (Fig 1B). 3 stores coupled (growth_fraction, rna_counts, protein_counts). Achieved 3.'} |
| cell-is-viable | kind=derived_scalar field=growth_fraction_final | op range low 0.9 high 1.1 provenance {'kind': 'model', 'note': 'Unperturbed integrated cell holds feasible FBA growth (growth_fraction ≈ 1). Achieved 1.000.'} |


## Study: Fig 2 — Cell growth: 9 h doubling time and protein-dominant composition (`fig2-growth`)

**Question.** Does the reduced-but-genuine viva-Mgen cell (FBA metabolism over iPS189
driving mass accumulation) reproduce the trained whole-cell model's core
growth phenotype from Fig 2 — a ~9 h doubling time and a protein-dominant
dry-mass composition?

**Objective.** Run the fig2_growth composite (metabolism + mass) for one cell cycle, fit the
exponential growth rate to recover the doubling time, confirm mass doubles over
the cycle, and read out the dry-mass composition at division.

**Hypothesis.** With growth calibrated to the model's fitted cell-cycle length (32400 s) and
the fitted dry-mass fractions from parameters.json, the integrated cell should
double its mass in ~9 h and show ~62% protein by dry mass, matching Fig 2A/B
(doubling time) and Fig 2C (composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig2-growth ===
STUDY = 'fig2-growth'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**growth-curve**


In [ ]:
# growth-curve
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**composition-donut**


In [ ]:
# composition-donut
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| doubling-time-near-9h | kind=derived_scalar field=doubling_time_h | op range low 8.5 high 9.5 provenance {'kind': 'experiment', 'note': 'M. genitalium doubling time τ ≈ 9 h (Karr 2012 Fig 2A/B); achieved 9.000 h.'} |
| mass-doubles-in-cycle | kind=derived_scalar field=final_mass_ratio | op range low 1.9 high 2.1 provenance {'kind': 'theory', 'note': 'A cell doubles its mass per cycle; achieved 2.000.'} |
| protein-dominant-composition | kind=derived_scalar field=protein_fraction | op range low 0.55 high 0.68 provenance {'kind': 'experiment', 'note': 'Fitted protein fraction 0.620 (Fig 2C).'} |


## Study: Fig 3 (2G/2H) — Single-cell expression: bursty mRNA vs accumulating protein (`fig3-expression`)

**Question.** Does the reduced viva-Mgen expression module (stochastic transcription +
translation + RNA/protein decay on a representative gene panel) reproduce the
qualitative single-cell gene-expression dynamics of Fig 3 / 2G-2H — bursty,
low-copy mRNA alongside protein that accumulates to much higher, more stable
copy numbers (the mRNA↔protein decoupling)?

**Objective.** Run the fig3_expression composite (transcription + translation + rna_decay +
protein_decay) for one hour at the default 1 s interval, gather per-gene mRNA
and protein counts, and read out mean mRNA per gene, final total protein, the
protein-to-mRNA ratio, and the fraction of panel genes expressed.

**Hypothesis.** With Poisson synthesis and Poisson decay running through shared stores, mRNA
should stay low and bursty (order 0-2 copies per gene, set by short half-lives
and low synthesis rates), while protein — synthesized proportionally to mRNA
and decaying far more slowly — should accumulate to copy numbers orders of
magnitude above mRNA, reproducing the Fig 2H decoupling qualitatively.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | seed=0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig3-expression ===
STUDY = 'fig3-expression'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**expression-timeseries**


In [ ]:
# expression-timeseries
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**protein-vs-mrna**


In [ ]:
# protein-vs-mrna
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| mrna-is-bursty-low-copy | kind=derived_scalar field=mean_mrna_per_gene | op range low 0.0 high 5.0 provenance {'kind': 'experiment', 'note': 'Single-cell mRNA is low-copy and bursty (Karr 2012 Fig 2G); reduced panel achieved mean_mrna_per_gene ~1.6. Representative rates, not KB-fitted.'} |
| protein-exceeds-mrna | kind=derived_scalar field=protein_to_mrna_ratio | op range low 5.0 high 1.0e9 provenance {'kind': 'experiment', 'note': 'mRNA and protein copy-number distributions are decoupled, protein far higher (Karr 2012 Fig 2H); achieved ratio ~200 (varies run to run).'} |
| genes-become-expressed | kind=derived_scalar field=fraction_genes_expressed | op range low 0.9 high 1.0 provenance {'kind': 'model', 'note': 'Over 1 h all representative genes express; achieved 1.000.'} |


## Study: Fig 4 — Emergent, unregulated control of cell-cycle duration (`fig4-cell-cycle`)

**Question.** Does the viva-Mgen replication submodel reproduce Fig 4's central finding —
that M. genitalium's cell-cycle duration is controlled *emergently*, without a
dedicated genetic regulator, through the coupling of replication initiation and
a dNTP surplus that it builds up?

**Objective.** Simulate a population of ~128 single cells with random birth DnaA and dNTP
levels by stepping the ReplicationReproductionProcess directly, recover the
three Fig 4 single-cell correlations, and render a representative single-cell
dynamics trajectory through the fig4_cell_cycle composite.

**Hypothesis.** If DnaA accumulates stochastically to trigger initiation while dNTPs
accumulate (unconsumed) during that same initiation phase, then across single
cells (1) more birth DnaA shortens initiation (Fig 4C), (2) a larger dNTP pool
at replication start shortens replication (Fig 4D), and (3) initiation and
replication durations are inversely correlated (Fig 4E) — a longer initiation
buys a bigger dNTP surplus that speeds replication, buffering total cycle length.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | initial_dnaA=5, initial_dntp=0, seed=0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig4-cell-cycle ===
STUDY = 'fig4-cell-cycle'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**init-vs-repl**


In [ ]:
# init-vs-repl
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**dntp-vs-repl**


In [ ]:
# dntp-vs-repl
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**cell-cycle-trajectory**


In [ ]:
# cell-cycle-trajectory
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| initiation-replication-inversely-correlated | kind=derived_scalar field=r_init_repl | op range low -1.0 high -0.7 provenance {'kind': 'model', 'note': 'Karr 2012 Fig 4E — inverse initiation↔replication duration control; achieved r = -0.930.'} |
| dntp-controls-replication-duration | kind=derived_scalar field=r_dntp_repl | op range low -1.0 high -0.7 provenance {'kind': 'model', 'note': 'Karr 2012 Fig 4D — higher starting dNTP shortens replication; achieved r = -0.951.'} |
| more-dnaa-shortens-initiation | kind=derived_scalar field=r_dnaA_init | op range low -1.0 high -0.4 provenance {'kind': 'model', 'note': 'Karr 2012 Fig 4C — more initial DnaA shortens initiation; achieved r = -0.743.'} |


## Study: Fig 5 — Global distribution of cellular energy: ATP > GTP synthesis and a translation-dominated budget (`fig5-energy`)

**Question.** Does the reduced viva-Mgen cell reproduce Fig 5's global energy picture — that
ATP and GTP are the dominant synthesized carriers with ATP > GTP (Fig 5A), and
that the cellular energy budget is dominated by translation, ahead of
transcription (Fig 5D)?

**Objective.** Run the fig5_energy composite for one hour (metabolism/mass at 60 s,
expression at 1 s) to time-average ATP/GTP production, then step the
transcription and translation processes directly for one hour to tally their
NTP/GTP consumption, and compute the modeled energy shares and the ATP:GTP
synthesis ratio.

**Hypothesis.** FBA over iPS189 should carry far more flux through ATP synthase (ATPS4r) than
through the GTP-producing kinases, giving ATP > GTP > 0. And because each
peptide bond costs ~2 GTP and translation events vastly outnumber transcription
events (mRNA accumulates and each transcript is translated many times over),
the modeled expression energy should be overwhelmingly translation, matching
the ordering translation >> transcription in Fig 5D.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig5-energy ===
STUDY = 'fig5-energy'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**energy-allocation**


In [ ]:
# energy-allocation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**synthesis-rates**


In [ ]:
# synthesis-rates
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| atp-exceeds-gtp-synthesis | kind=derived_scalar field=atp_to_gtp_ratio | op range low 2.0 high 8.0 provenance {'kind': 'model', 'note': 'Fig 5A shows ATP and GTP as the dominant synthesized carriers with ATP > GTP; iPS189 FBA gives ratio 4.03 (ATP 0.813 vs GTP 0.202).'} |
| translation-dominates-energy | kind=derived_scalar field=translation_share | op range low 0.5 high 1.0 provenance {'kind': 'model', 'note': "Fig 5D's budget is translation-dominated (~29%, ahead of transcription ~7%); the reduced model reproduces the ordering with translation at 0.968 of the modeled expression energy."} |


## Study: Fig 6A — Single-gene-disruption essentiality: model vs experiment (`fig6-gene-essentiality`)

**Question.** Does the viva-Mgen FBA metabolism submodel reproduce Fig 6A — the
single-gene-disruption essentiality phenotype — for the metabolic genes of
M. genitalium: does an in-silico single-gene knockout predict which genes are
essential, matching the reference essentiality calls?

**Objective.** For every metabolic gene with a reference call, run a single-gene FBA knockout
via MetabolismFbaReproductionProcess, call it essential when growth_fraction
drops below 0.05, cross-tabulate against the reference calls into a 2x2
confusion matrix, and score accuracy, sensitivity, and specificity. Also run
the fig6_gene_essentiality composite once for a representative essential gene
(MG_023) to record a canonical baseline and confirm growth collapses.

**Hypothesis.** For the ~125 genes the iPS189 metabolic reconstruction covers (the paper notes
metabolic-gene disruptions are the most debilitating), a single-gene FBA
knockout that collapses growth should mark the gene essential, and the
model-vs-reference confusion matrix should land near the paper's whole-cell
accuracy (~79% overall; ~87% for the iPS189 metabolic model on essentiality).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig6-gene-essentiality ===
STUDY = 'fig6-gene-essentiality'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**confusion-matrix**


In [ ]:
# confusion-matrix
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**accuracy-summary**


In [ ]:
# accuracy-summary
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| essentiality-accuracy-high | kind=derived_scalar field=essentiality_accuracy | op range low 0.7 high 1.0 provenance {'kind': 'experiment', 'note': 'Gene-essentiality accuracy: 79% overall for the Karr 2012 whole-cell model (Fig 6A; Glass et al. 2006 essentiality data), ~87% for the iPS189 metabolic model (Suthers et al. 2009). Achieved 0.760 over 125 metabolic genes with a reference call.'} |
| essential-genes-detected | kind=derived_scalar field=sensitivity | op range low 0.7 high 1.0 provenance {'kind': 'experiment', 'note': 'Reference-essential genes should collapse growth on single knockout. Achieved sensitivity 0.755 (80 of 106 reference-essential genes detected).'} |


## Study: Fig 7E-G — Kinetic parameters: growth depends sigmoidally on an enzyme's kcat (Vmax proxy) (`fig7-kinetic-parameters`)

**Question.** Does the reduced-but-genuine viva-Mgen cell reproduce Fig 7E-G's central
quantitative claim — that predicted growth rate depends on an enzyme's kinetic
parameter (kcat/Vmax), producing a SIGMOIDAL growth-vs-kcat curve that rises
from near-zero at low activity and saturates at the wild-type rate once the
enzyme is no longer rate-limiting?

**Objective.** Programmatically identify a growth-limiting reaction in the iPS189 network
(one whose bound binds when throttled yet whose growth recovers to the
wild-type plateau before the top of the sweep), sweep its flux bound over
np.logspace(-2, 1, 16) with MetabolismFbaReproductionProcess, and read out
whether the kcat→growth curve is monotonic, saturating, and has a clear
dynamic range.

**Hypothesis.** If a genuinely growth-limiting reaction's FBA flux bound is used as a kcat
proxy and swept over ~three decades (0.01–10x wild-type), growth_fraction
should rise monotonically with the bound and then plateau at 1.0, mirroring the
saturating kcat→growth dependence the paper uses to reconcile model and
experiment for lpdA/deoD/thyA.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.mgen.mycoplasma_genitalium` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.mgen.mycoplasma_genitalium`** — `spec_viva_mgen_composites_mgen_mycoplasma_genitalium` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.mgen.mycoplasma_genitalium` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig7-kinetic-parameters ===
STUDY = 'fig7-kinetic-parameters'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**kcat-growth-curve**


In [ ]:
# kcat-growth-curve
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| growth-monotonic-in-kcat | kind=derived_scalar field=growth_is_monotonic_in_kcat | op range low 1.0 high 1.0 provenance {'kind': 'theory', 'note': 'Relaxing a rate-limiting Vmax cannot decrease FBA growth; the kcat→growth curve is monotonic (Karr 2012 Fig 7E). Achieved 1.0.'} |
| growth-saturates-at-high-kcat | kind=derived_scalar field=growth_saturates | op range low 1.0 high 1.0 provenance {'kind': 'theory', 'note': 'Past the point where the enzyme stops limiting flux, growth plateaus at the wild-type rate (Karr 2012 Fig 7E-G). Achieved 1.0.'} |
| kcat-has-dynamic-range | kind=derived_scalar field=dynamic_range | op range low 0.3 high 1.05 provenance {'kind': 'model', 'note': 'EX_leu_DASH_L_e is genuinely growth-limiting; sweeping its bound moves growth from 0.136 to 1.000. Achieved dynamic_range 0.864.'} |
